# Phase 1 — Image-Only Baseline
8. Extract frozen EyeCLIP embeddings (full dataset)
9. Linear classifier on frozen embeddings (8-class)
10. Evaluate: accuracy, macro-F1, precision, recall, confusion matrix, per-class F1

**Assumptions (change if wrong):**
- `train.csv` / `val.csv` / `test.csv` each have columns: `image_path`, `patient_id`, `label`
- `image_path` is relative to `IMAGE_ROOT`
- `label` is a string class name (8 classes)

In [1]:
%pip install -q setuptools packaging ftfy regex tqdm scikit-learn
%pip install -q git+https://github.com/openai/CLIP.git
!git clone --depth 1 https://github.com/Michi-3000/EyeCLIP.git || true

import pathlib
clip_path = pathlib.Path("EyeCLIP/eyeclip/clip.py")
text = clip_path.read_text()
text = text.replace("from pkg_resources import packaging", "import packaging")
clip_path.write_text(text)

import sys
sys.path.insert(0, "EyeCLIP")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.
Cloning into 'EyeCLIP'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 30 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 1.34 MiB | 1.89 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [2]:
import torch, eyeclip
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_ROOT = Path("/kaggle/input/datasets/faruq05/rop-vl/ROP-VL/Image")   # CHANGE to actual path
SPLIT_DIR = Path("/kaggle/input/datasets/faruq05/rop-vl/ROP-VL/split")    # CHANGE to actual path
CKPT_PATH = "/kaggle/input/datasets/faruq05/rop-vl/ROP-VL/eyeclip.pt"                    # CHANGE to actual path

model, preprocess = eyeclip.load("ViT-B/32", device=DEVICE, jit=False)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
state = ckpt.get("model_state_dict") or ckpt.get("state_dict") or ckpt
model.load_state_dict(state, strict=False)
model.eval()
for p in model.parameters():
    p.requires_grad = False
print("EyeCLIP loaded on", DEVICE)

100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 121MiB/s]


EyeCLIP loaded on cuda


In [3]:
RENAME = {"image": "image_path", "categories": "label"}

train_df = pd.read_csv(SPLIT_DIR / "train.csv").rename(columns=RENAME)
val_df   = pd.read_csv(SPLIT_DIR / "val.csv").rename(columns=RENAME)
test_df  = pd.read_csv(SPLIT_DIR / "test.csv").rename(columns=RENAME)

# leak check: no patient_id should appear in more than one split
train_p, val_p, test_p = set(train_df.patient_id), set(val_df.patient_id), set(test_df.patient_id)
assert not (train_p & val_p), "patient leak: train/val"
assert not (train_p & test_p), "patient leak: train/test"
assert not (val_p & test_p), "patient leak: val/test"
print("no patient leakage across splits")

classes = sorted(train_df.label.unique())
assert len(classes) == 8, f"expected 8 classes, got {len(classes)}"
label2idx = {c: i for i, c in enumerate(classes)}
print("classes:", classes)

no patient leakage across splits
classes: ['A-ROP', 'Laser-treated ROP', 'Stage 1 ROP', 'Stage 2 ROP', 'Stage 3 ROP', 'Stage 4 ROP', 'Stage 5 ROP', 'normal']


In [4]:
# map filename -> full path by scanning all subfolders
path_lookup = {p.name: p for p in IMAGE_ROOT.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
print("images found on disk:", len(path_lookup))

# check for duplicate filenames across folders (would silently overwrite in the dict)
n_files = sum(1 for p in IMAGE_ROOT.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print("total files:", n_files, "| unique names:", len(path_lookup))

# resolve every split
for df in (train_df, val_df, test_df):
    df["full_path"] = df["image_path"].map(path_lookup)
    print("missing:", df["full_path"].isna().sum(), "of", len(df))

images found on disk: 2020
total files: 2020 | unique names: 2020
missing: 0 of 1411
missing: 0 of 289
missing: 0 of 320


In [5]:
class ROPImageDataset(Dataset):
    """Maps (image_path, label) rows -> (preprocessed tensor, label idx)."""
    def __init__(self, df, image_root, preprocess, label2idx):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.preprocess = preprocess
        self.label2idx = label2idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row.full_path).convert("RGB")
        x = self.preprocess(img)
        y = self.label2idx[row.label]
        return x, y

In [6]:
@torch.no_grad()
def extract_embeddings(df, batch_size=64, num_workers=2):
    """df -> (embeddings [N,512] float32 np.ndarray, labels [N] int np.ndarray)"""
    ds = ROPImageDataset(df, IMAGE_ROOT, preprocess, label2idx)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    feats, labels = [], []
    for x, y in dl:
        x = x.to(DEVICE)
        emb = model.encode_image(x).float().cpu().numpy()
        feats.append(emb)
        labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

X_train, y_train = extract_embeddings(train_df)
X_val,   y_val   = extract_embeddings(val_df)
X_test,  y_test  = extract_embeddings(test_df)

print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)
np.savez("eyeclip_embeddings.npz",
         X_train=X_train, y_train=y_train,
         X_val=X_val, y_val=y_val,
         X_test=X_test, y_test=y_test)

train: (1411, 512) val: (289, 512) test: (320, 512)


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
clf.fit(X_train_s, y_train)

val_acc = clf.score(X_val_s, y_val)
print("val accuracy:", val_acc)

val accuracy: 0.31833910034602075


In [8]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              classification_report, confusion_matrix)

y_pred = clf.predict(X_test_s)

acc = accuracy_score(y_test, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, average=None, labels=range(len(classes)), zero_division=0)
macro_f1 = f1.mean()

print(f"Test accuracy: {acc:.4f}")
print(f"Macro-F1:      {macro_f1:.4f}\n")

print("Per-class (support = # test images per class, flags imbalance):")
for c, p, r, f, s in zip(classes, precision, recall, f1, support):
    print(f"  {c:15s}  P={p:.3f}  R={r:.3f}  F1={f:.3f}  n={s}")

print("\nFull report:")
print(classification_report(y_test, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=range(len(classes)))
cm_df = pd.DataFrame(cm, index=classes, columns=classes)
print("\nConfusion matrix:")
print(cm_df)
cm_df.to_csv("phase1_confusion_matrix.csv")

Test accuracy: 0.3063
Macro-F1:      0.2469

Per-class (support = # test images per class, flags imbalance):
  A-ROP            P=0.159  R=0.280  F1=0.203  n=25
  Laser-treated ROP  P=0.547  R=0.537  F1=0.542  n=54
  Stage 1 ROP      P=0.327  R=0.432  F1=0.372  n=37
  Stage 2 ROP      P=0.286  R=0.231  F1=0.255  n=52
  Stage 3 ROP      P=0.086  R=0.079  F1=0.082  n=38
  Stage 4 ROP      P=0.158  R=0.250  F1=0.194  n=12
  Stage 5 ROP      P=0.000  R=0.000  F1=0.000  n=5
  normal           P=0.378  R=0.289  F1=0.327  n=97

Full report:
                   precision    recall  f1-score   support

            A-ROP       0.16      0.28      0.20        25
Laser-treated ROP       0.55      0.54      0.54        54
      Stage 1 ROP       0.33      0.43      0.37        37
      Stage 2 ROP       0.29      0.23      0.26        52
      Stage 3 ROP       0.09      0.08      0.08        38
      Stage 4 ROP       0.16      0.25      0.19        12
      Stage 5 ROP       0.00      0.00      0.

## Done — Phase 1 deliverable
Record `acc` and `macro_f1` above as the baseline number.
Do **not** proceed to few-shot prototypes (Phase 2) in this notebook.